# 01 - Ingestion Layer: Kafka + Schema Contract (Rubric item 1 / 20 pts)

**Project:** ShopSense - a real-time e-commerce lakehouse + RAG assistant
**Program:** SDAIA Academy - Modern Data Engineering for AI Systems (5-day capstone)

## What this notebook must prove
| Rubric requirement | Where it is proven |
|---|---|
| A **real** Kafka broker (not a queue simulation) | Section 2 - Kafka 3.7.1 in KRaft mode running on `localhost:9092` |
| Kafka **producer** | Section 5 |
| Kafka **consumer** | Section 6 |
| **Schema validation at the ingestion boundary** (Pydantic data contract) | Section 4 - `OrderEvent` model |
| Malformed records routed to a **dead-letter topic / quarantine zone** | Section 6 - topic `orders.dlq` + `data/quarantine/` |
| **Rejection reason recorded** | `rejection_reason` field on every DLQ message |
| Failure path proven, not just the happy path | Section 7 - we read the DLQ back and print the rejected records |

> Run every cell top to bottom **and keep the output**. The saved output is the evidence.

## 1. Setup - Google Drive + project folders

Colab wipes `/content` when the session ends, and each notebook runs on a different machine.
So everything we want to keep (and hand to notebook 02) lives in Google Drive.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pathlib import Path
import os, json, datetime

PROJECT = Path('/content/drive/MyDrive/sdaia_capstone')
DATA    = PROJECT / 'data'
LANDING = DATA / 'bronze_landing'      # valid records -> input for notebook 02
QUARANT = DATA / 'quarantine'          # rejected records (quarantine zone)
REPORTS = PROJECT / 'reports'

for p in [PROJECT, DATA, LANDING, QUARANT, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT)
for p in sorted(PROJECT.iterdir()):
    print('  ', p.name)

Project root : /content/drive/MyDrive/sdaia_capstone
   README.md
   START_HERE_ar.md
   data
   docs
   gitignore
   notebooks
   reports


## 2. Install and start a real Apache Kafka broker

We run **Kafka 3.7.1 in KRaft mode** (no ZooKeeper) directly inside the Colab VM.
This is a genuine broker speaking the real Kafka wire protocol - not a Python queue standing in for Kafka.

In [10]:
# Java is required by the Kafka broker
!java -version 2>&1 | head -1 || apt-get -qq install -y openjdk-17-jdk-headless

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [11]:
%%bash
set -e
cd /content
if [ ! -d kafka_2.13-3.7.1 ]; then
  wget -q https://archive.apache.org/dist/kafka/3.7.1/kafka_2.13-3.7.1.tgz
  tar -xzf kafka_2.13-3.7.1.tgz
fi
ls -d /content/kafka_2.13-3.7.1

/content/kafka_2.13-3.7.1


In [12]:
import base64, uuid, subprocess

KAFKA = '/content/kafka_2.13-3.7.1'

# Generate the cluster id in Python instead of shell command substitution:
# the JVM prints cgroup warnings on stdout, which corrupted the shell variable.
CLUSTER_ID = base64.urlsafe_b64encode(uuid.uuid4().bytes).decode().rstrip('=')
print('Cluster id:', CLUSTER_ID)

fmt = subprocess.run(
    [f'{KAFKA}/bin/kafka-storage.sh', 'format',
     '-t', CLUSTER_ID,
     '-c', f'{KAFKA}/config/kraft/server.properties',
     '--ignore-formatted'],
    capture_output=True, text=True)
print((fmt.stdout or fmt.stderr).strip()[-400:])

log = open('/content/kafka.log', 'w')
subprocess.Popen([f'{KAFKA}/bin/kafka-server-start.sh',
                  f'{KAFKA}/config/kraft/server.properties'],
                 stdout=log, stderr=subprocess.STDOUT)
print('broker starting...')

Cluster id: nzLRFSjUQqa7RRfJQGutUQ
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
broker starting...


In [13]:
import time, subprocess
KAFKA = '/content/kafka_2.13-3.7.1'

# Wait until the broker answers on 9092
for attempt in range(30):
    r = subprocess.run([f'{KAFKA}/bin/kafka-broker-api-versions.sh',
                        '--bootstrap-server', 'localhost:9092'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Kafka broker is UP after {attempt*3}s')
        print(r.stdout.splitlines()[0])
        break
    time.sleep(3)
else:
    print('Broker did not start - last 30 lines of the log:')
    print(open('/content/kafka.log').read()[-3000:])

Kafka broker is UP after 0s
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


### 2.1 Create the topics

| Topic | Purpose |
|---|---|
| `orders.raw` | everything the producer emits, good and bad |
| `orders.valid` | records that passed the data contract |
| `orders.dlq` | **dead-letter topic** - rejected records + the reason they were rejected |

In [14]:
%%bash
KAFKA=/content/kafka_2.13-3.7.1
for T in orders.raw orders.valid orders.dlq; do
  $KAFKA/bin/kafka-topics.sh --create --if-not-exists --topic $T \
      --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
done
echo "--- topics on the broker ---"
$KAFKA/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

[0.006s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.006s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Created topic orders.raw.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Created topic orders.valid.
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-chil

## 3. Python client libraries

In [15]:
!pip install -q "kafka-python>=2.2.0" "pydantic>=2.7" faker
import kafka, pydantic
print('kafka-python', kafka.__version__, '| pydantic', pydantic.VERSION)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.9 MB/s eta 0:00:00
kafka-python 3.0.11 | pydantic 2.13.4


## 4. The data contract (schema validation at the ingestion boundary)

`OrderEvent` is the **contract**: any message that does not satisfy it never reaches Bronze.
Note the validators - they catch things a plain type check would let through
(negative quantities, unsupported currency, a delivery date before the order date).

In [16]:
from __future__ import annotations
from datetime import datetime, timezone, timedelta
from typing import Literal, Optional
from pydantic import BaseModel, Field, EmailStr, field_validator, model_validator, ConfigDict

ALLOWED_CATEGORIES = {'electronics', 'grocery', 'fashion', 'home', 'beauty', 'sports'}

class OrderEvent(BaseModel):
    # Data contract for a single order event arriving on `orders.raw`.
    model_config = ConfigDict(extra='forbid', str_strip_whitespace=True)

    order_id:       str   = Field(..., min_length=6, max_length=32)
    customer_id:    str   = Field(..., min_length=4, max_length=32)
    customer_email: str   = Field(..., max_length=120)
    product_id:     str   = Field(..., min_length=3)
    product_name:   str   = Field(..., min_length=2, max_length=120)
    category:       str
    quantity:       int   = Field(..., gt=0, le=100)
    unit_price:     float = Field(..., gt=0)
    currency:       Literal['SAR', 'USD', 'AED']
    city:           str   = Field(..., min_length=2, max_length=60)
    status:         Literal['created', 'paid', 'shipped', 'delivered', 'cancelled']
    order_ts:       datetime
    delivered_ts:   Optional[datetime] = None

    @field_validator('category')
    @classmethod
    def category_must_be_known(cls, v: str) -> str:
        v = v.lower()
        if v not in ALLOWED_CATEGORIES:
            raise ValueError(f'unknown category {v!r}; allowed: {sorted(ALLOWED_CATEGORIES)}')
        return v

    @field_validator('customer_email')
    @classmethod
    def email_shape(cls, v: str) -> str:
        if '@' not in v or '.' not in v.split('@')[-1]:
            raise ValueError(f'malformed email address: {v!r}')
        return v.lower()

    @model_validator(mode='after')
    def delivery_after_order(self):
        if self.delivered_ts is not None and self.delivered_ts < self.order_ts:
            raise ValueError('delivered_ts is earlier than order_ts')
        return self

print(json.dumps(OrderEvent.model_json_schema(), indent=2)[:900], '...')

{
  "additionalProperties": false,
  "properties": {
    "order_id": {
      "maxLength": 32,
      "minLength": 6,
      "title": "Order Id",
      "type": "string"
    },
    "customer_id": {
      "maxLength": 32,
      "minLength": 4,
      "title": "Customer Id",
      "type": "string"
    },
    "customer_email": {
      "maxLength": 120,
      "title": "Customer Email",
      "type": "string"
    },
    "product_id": {
      "minLength": 3,
      "title": "Product Id",
      "type": "string"
    },
    "product_name": {
      "maxLength": 120,
      "minLength": 2,
      "title": "Product Name",
      "type": "string"
    },
    "category": {
      "title": "Category",
      "type": "string"
    },
    "quantity": {
      "exclusiveMinimum": 0,
      "maximum": 100,
      "title": "Quantity",
      "type": "integer"
    },
    "unit_price": {
      "exclusiveMinimum": 0,
      "ti ...


In [17]:
# Save the contract next to the data - the schema is a project artefact, not a hidden detail
(DATA / 'contracts').mkdir(exist_ok=True)
(DATA / 'contracts' / 'order_event.schema.json').write_text(
    json.dumps(OrderEvent.model_json_schema(), indent=2)
)
print('contract written ->', DATA / 'contracts' / 'order_event.schema.json')

contract written -> /content/drive/MyDrive/sdaia_capstone/data/contracts/order_event.schema.json


## 5. Producer - emit good **and** deliberately broken events

A capstone that only produces clean data cannot prove the rejection path.
About 22% of what we emit here is broken on purpose, one defect per record so the
rejection reasons stay readable.

In [18]:
import random, uuid
from faker import Faker

fake = Faker()
random.seed(42); Faker.seed(42)

CITIES  = ['Riyadh', 'Buraydah', 'Jeddah', 'Dammam', 'Abha', 'Madinah', 'Unaizah']
CATALOG = [
    ('P-1001', 'Wireless Earbuds',   'electronics', 249.0),
    ('P-1002', 'Smart Watch',        'electronics', 899.0),
    ('P-2001', 'Olive Oil 1L',       'grocery',      42.5),
    ('P-2002', 'Arabic Coffee 500g', 'grocery',      35.0),
    ('P-3001', 'Cotton Abaya',       'fashion',     320.0),
    ('P-3002', 'Running Shoes',      'sports',      410.0),
    ('P-4001', 'Air Fryer',          'home',        520.0),
    ('P-5001', 'Face Serum',         'beauty',      180.0),
]

def make_valid_event():
    pid, pname, cat, price = random.choice(CATALOG)
    order_ts = fake.date_time_between(start_date='-14d', end_date='now', tzinfo=timezone.utc)
    status   = random.choice(['created', 'paid', 'shipped', 'delivered', 'cancelled'])
    delivered = None
    if status == 'delivered':
        delivered = order_ts + timedelta(days=random.randint(1, 6))
    return {
        'order_id':       f'ORD-{uuid.uuid4().hex[:10].upper()}',
        'customer_id':    f'CUST-{random.randint(1000, 1120)}',
        'customer_email': fake.email(),
        'product_id':     pid,
        'product_name':   pname,
        'category':       cat,
        'quantity':       random.randint(1, 5),
        'unit_price':     price,
        'currency':       'SAR',
        'city':           random.choice(CITIES),
        'status':         status,
        'order_ts':       order_ts.isoformat(),
        'delivered_ts':   delivered.isoformat() if delivered else None,
    }

DEFECTS = [
    ('missing_required_field',  lambda e: e.pop('customer_id')),
    ('negative_quantity',       lambda e: e.update(quantity=-3)),
    ('zero_price',              lambda e: e.update(unit_price=0)),
    ('wrong_type_quantity',     lambda e: e.update(quantity='three')),
    ('unknown_category',        lambda e: e.update(category='furniture')),
    ('unsupported_currency',    lambda e: e.update(currency='EUR')),
    ('malformed_email',         lambda e: e.update(customer_email='ali[at]shop')),
    ('invalid_status',          lambda e: e.update(status='returned')),
    ('undeclared_extra_field',  lambda e: e.update(discount_hack=True)),
    ('delivered_before_order',  lambda e: e.update(status='delivered',
                                                  delivered_ts='2020-01-01T00:00:00+00:00')),
]

def make_broken_event():
    e = make_valid_event()
    name, defect = random.choice(DEFECTS)
    defect(e)
    e['_injected_defect'] = name      # for our own reporting only
    return e

N_EVENTS   = 400
BROKEN_PCT = 0.22

events = []
for _ in range(N_EVENTS):
    events.append(make_broken_event() if random.random() < BROKEN_PCT else make_valid_event())

# a few duplicates + status updates so notebook 02 has something real to MERGE on
for e in random.sample([e for e in events if '_injected_defect' not in e], 25):
    upd = dict(e); upd['status'] = 'delivered'
    upd['delivered_ts'] = (datetime.fromisoformat(e['order_ts']) + timedelta(days=2)).isoformat()
    events.append(upd)

print('events to produce :', len(events))
print('deliberately broken:', sum(1 for e in events if '_injected_defect' in e))
print('\nsample valid event:'); print(json.dumps(events[0], indent=2))

events to produce : 425
deliberately broken: 78

sample valid event:
{
  "order_id": "ORD-7E35AF7810",
  "customer_id": "CUST-1031",
  "customer_email": "figueroajohn@example.org",
  "product_id": "P-1001",
  "product_name": "Wireless Earbuds",
  "category": "electronics",
  "quantity": 2,
  "unit_price": 249.0,
  "currency": "SAR",
  "city": "Buraydah",
  "status": "shipped",
  "order_ts": "2026-09-03T18:32:04.655415+00:00",
  "delivered_ts": null
}


In [19]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    acks='all',            # wait for the broker to acknowledge
    retries=3,
)

for e in events:
    producer.send('orders.raw', key=e.get('order_id'), value=e)
producer.flush()

print(f'produced {len(events)} messages to topic orders.raw')
print('producer metrics:', {k: v for k, v in list(producer.metrics()['producer-metrics'].items())[:3]})
producer.close()

produced 425 messages to topic orders.raw
producer metrics: {'connection-close-rate': 0.032881109038243574, 'connection-creation-rate': 0.06573802922388827, 'connection-count': 1.0}


## 6. Consumer - validate at the boundary, quarantine what fails

This is the heart of deliverable 1. For every message:

1. parse the JSON,
2. validate against the `OrderEvent` contract,
3. **pass** -> publish to `orders.valid` and append to the Bronze landing file,
4. **fail** -> publish to `orders.dlq` **with the rejection reason** and append to the quarantine file.

The pipeline never crashes on bad data and never silently drops it.

In [20]:
from kafka import KafkaConsumer
from pydantic import ValidationError

consumer = KafkaConsumer(
    'orders.raw',
    bootstrap_servers=['localhost:9092'],
    group_id='ingestion-validator-v1',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda b: b.decode('utf-8'),
    consumer_timeout_ms=15000,     # stop when the topic is drained
)

dlq_producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
)

run_id     = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
valid_path = LANDING / f'orders_valid_{run_id}.jsonl'
quar_path  = QUARANT / f'orders_rejected_{run_id}.jsonl'

n_ok = n_bad = 0
reason_counts = {}

with open(valid_path, 'w') as f_ok, open(quar_path, 'w') as f_bad:
    for msg in consumer:
        raw_text = msg.value
        meta = {'kafka_topic': msg.topic, 'kafka_partition': msg.partition,
                'kafka_offset': msg.offset, 'ingested_at': datetime.now(timezone.utc).isoformat()}
        try:
            payload = json.loads(raw_text)
        except json.JSONDecodeError as exc:
            reason = f'invalid_json: {exc}'
            payload = None
        else:
            reason = None

        if payload is not None:
            candidate = {k: v for k, v in payload.items() if not k.startswith('_')}
            try:
                event = OrderEvent(**candidate)
            except ValidationError as exc:
                first = exc.errors()[0]
                reason = f"{'.'.join(str(x) for x in first['loc']) or 'record'}: {first['msg']}"
            else:
                record = event.model_dump(mode='json') | meta
                f_ok.write(json.dumps(record) + '\n')
                dlq_producer.send('orders.valid', value=record)
                n_ok += 1
                continue

        rejected = {'rejection_reason': reason,
                    'raw_payload': raw_text,
                    **meta}
        f_bad.write(json.dumps(rejected) + '\n')
        dlq_producer.send('orders.dlq', value=rejected)
        n_bad += 1
        key = reason.split(':')[0].strip()
        reason_counts[key] = reason_counts.get(key, 0) + 1

    consumer.commit()

dlq_producer.flush(); dlq_producer.close(); consumer.close()

total = n_ok + n_bad
print(f'consumed        : {total}')
print(f'accepted (bronze): {n_ok}   ({n_ok/total:.1%})')
print(f'rejected  (dlq)  : {n_bad}   ({n_bad/total:.1%})')
print('\nrejections by field / rule:')
for k, v in sorted(reason_counts.items(), key=lambda kv: -kv[1]):
    print(f'  {v:>4}  {k}')
print('\nbronze landing file  ->', valid_path)
print('quarantine file      ->', quar_path)

/tmp/ipykernel_3116/104309017.py:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


consumed        : 425
accepted (bronze): 347   (81.6%)
rejected  (dlq)  : 78   (18.4%)

rejections by field / rule:
    13  status
    12  currency
    10  unit_price
    10  record
     7  category
     7  customer_email
     7  customer_id
     6  discount_hack
     6  quantity

bronze landing file  -> /content/drive/MyDrive/sdaia_capstone/data/bronze_landing/orders_valid_20260908T194115Z.jsonl
quarantine file      -> /content/drive/MyDrive/sdaia_capstone/data/quarantine/orders_rejected_20260908T194115Z.jsonl


## 7. Proof of the failure path

The rubric asks for malformed records *actually* being rejected. We now read the
**dead-letter topic back off the broker** and show what is in it.

In [21]:
dlq_consumer = KafkaConsumer(
    'orders.dlq',
    bootstrap_servers=['localhost:9092'],
    group_id='dlq-inspector',
    auto_offset_reset='earliest',
    value_deserializer=lambda b: json.loads(b.decode('utf-8')),
    consumer_timeout_ms=10000,
)

dlq_rows = [m.value for m in dlq_consumer]
dlq_consumer.close()

print(f'messages sitting in orders.dlq: {len(dlq_rows)}\n')
for row in dlq_rows[:8]:
    print('REASON  :', row['rejection_reason'])
    print('OFFSET  :', f"{row['kafka_topic']}[{row['kafka_partition']}]@{row['kafka_offset']}")
    print('PAYLOAD :', row['raw_payload'][:160], '...')
    print('-' * 90)

/tmp/ipykernel_3116/781465335.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  dlq_consumer = KafkaConsumer(


messages sitting in orders.dlq: 78

REASON  : unit_price: Input should be greater than 0
OFFSET  : orders.raw[0]@9
PAYLOAD : {"order_id": "ORD-C96E926259", "customer_id": "CUST-1097", "customer_email": "nataliearroyo@example.org", "product_id": "P-5001", "product_name": "Face Serum",  ...
------------------------------------------------------------------------------------------
REASON  : category: Value error, unknown category 'furniture'; allowed: ['beauty', 'electronics', 'fashion', 'grocery', 'home', 'sports']
OFFSET  : orders.raw[0]@11
PAYLOAD : {"order_id": "ORD-EA782EFC9C", "customer_id": "CUST-1009", "customer_email": "adrianzimmerman@example.org", "product_id": "P-1001", "product_name": "Wireless Ea ...
------------------------------------------------------------------------------------------
REASON  : customer_email: Value error, malformed email address: 'ali[at]shop'
OFFSET  : orders.raw[0]@13
PAYLOAD : {"order_id": "ORD-088100CE44", "customer_id": "CUST-1021", "customer_emai

In [22]:
import pandas as pd
dlq_df = pd.DataFrame(dlq_rows)
dlq_df['rule'] = dlq_df['rejection_reason'].str.split(':').str[0]
summary = dlq_df['rule'].value_counts().rename_axis('failed_rule').reset_index(name='rejected_records')
display(summary)

,failed_rule,rejected_records
0,status,13
1,currency,12
2,unit_price,10
3,record,10
4,category,7
5,customer_id,7
6,customer_email,7
7,discount_hack,6
8,quantity,6


### 7.1 Confirm the same thing from the broker's own CLI (independent evidence)

In [23]:
%%bash
KAFKA=/content/kafka_2.13-3.7.1
echo "--- message counts per topic (end offsets) ---"
for T in orders.raw orders.valid orders.dlq; do
  echo -n "$T : "
  $KAFKA/bin/kafka-get-offsets.sh --bootstrap-server localhost:9092 --topic $T 2>/dev/null \
      | awk -F: '{s+=$3} END {print s}'
done

--- message counts per topic (end offsets) ---
orders.raw : 425
orders.valid : 347
orders.dlq : 78


## 8. Ingestion report - handed to notebook 02 and to the quality gate on day 4

In [24]:
report = {
    'run_id': run_id,
    'stage': 'ingestion',
    'broker': 'kafka 3.7.1 (KRaft) @ localhost:9092',
    'client': f'kafka-python {kafka.__version__}',
    'contract': 'OrderEvent (pydantic v2)',
    'topics': {'raw': 'orders.raw', 'valid': 'orders.valid', 'dead_letter': 'orders.dlq'},
    'messages_consumed': total,
    'accepted': n_ok,
    'rejected': n_bad,
    'acceptance_rate': round(n_ok / total, 4),
    'rejections_by_rule': reason_counts,
    'bronze_landing_file': str(valid_path),
    'quarantine_file': str(quar_path),
    'finished_at': datetime.now(timezone.utc).isoformat(),
}
out = REPORTS / f'ingestion_report_{run_id}.json'
out.write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print('\nsaved ->', out)

{
  "run_id": "20260908T194115Z",
  "stage": "ingestion",
  "broker": "kafka 3.7.1 (KRaft) @ localhost:9092",
  "client": "kafka-python 3.0.11",
  "contract": "OrderEvent (pydantic v2)",
  "topics": {
    "raw": "orders.raw",
    "valid": "orders.valid",
    "dead_letter": "orders.dlq"
  },
  "messages_consumed": 425,
  "accepted": 347,
  "rejected": 78,
  "acceptance_rate": 0.8165,
  "rejections_by_rule": {
    "unit_price": 10,
    "category": 7,
    "customer_email": 7,
    "record": 10,
    "customer_id": 7,
    "status": 13,
    "currency": 12,
    "discount_hack": 6,
    "quantity": 6
  },
  "bronze_landing_file": "/content/drive/MyDrive/sdaia_capstone/data/bronze_landing/orders_valid_20260908T194115Z.jsonl",
  "quarantine_file": "/content/drive/MyDrive/sdaia_capstone/data/quarantine/orders_rejected_20260908T194115Z.jsonl",
  "finished_at": "2026-09-08T19:41:57.734427+00:00"
}

saved -> /content/drive/MyDrive/sdaia_capstone/reports/ingestion_report_20260908T194115Z.json


## 9. What notebook 02 picks up

```
MyDrive/sdaia_capstone/
  data/
    contracts/order_event.schema.json
    bronze_landing/orders_valid_<run_id>.jsonl   <-- input to the Bronze layer
    quarantine/orders_rejected_<run_id>.jsonl    <-- quarantine zone
  reports/ingestion_report_<run_id>.json
```

**Before you close this notebook:** `Runtime -> Run all`, wait for it to finish, then
`File -> Save`. The saved outputs are what the trainer grades. Then push it to GitHub
(see `docs/github_guide_ar.md` in the repo).